# Workora A/B Testing & Hypothesis Testing

**Task:** Compare checkout conversion for Control A and Variant B.

Keep `checkout_ab_test_v2.csv` in the same folder as this notebook. Run cells from top to bottom.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, chi2_contingency

df = pd.read_csv('checkout_ab_test_v2.csv')

## 1. Inspect the dataset

In [ ]:
print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nMissing values:\n', df.isna().sum())
print('\nSample rows:')
display(df.head())

## 2. Calculate sample sizes and conversion rates

In [ ]:
summary = df.groupby('Variant').agg(
    visitors=('VisitorID', 'count'),
    checkout_starts=('CheckoutStarted', lambda s: (s == 'Yes').sum()),
    conversions=('Converted', lambda s: (s == 'Yes').sum()),
    revenue_inr=('OrderValueINR', 'sum')
)
summary['conversion_rate'] = summary['conversions'] / summary['visitors']
summary['checkout_start_rate'] = summary['checkout_starts'] / summary['visitors']
display(summary)

## 3. State the hypotheses

- **H₀:** Control A and Variant B have equal conversion rates.
- **H₁:** Control A and Variant B have different conversion rates.
- Significance level: **α = 0.05**.
- Use a two-sided two-proportion Z-test.

In [ ]:
nA = int(summary.loc['A', 'visitors'])
nB = int(summary.loc['B', 'visitors'])
xA = int(summary.loc['A', 'conversions'])
xB = int(summary.loc['B', 'conversions'])
pA = xA / nA
pB = xB / nB
pooled = (xA + xB) / (nA + nB)
se_pooled = np.sqrt(pooled * (1 - pooled) * (1/nA + 1/nB))
z = (pB - pA) / se_pooled
p_value = 2 * (1 - norm.cdf(abs(z)))
print(f'Control A conversion rate: {pA:.2%} ({xA}/{nA})')
print(f'Variant B conversion rate: {pB:.2%} ({xB}/{nB})')
print(f'Z statistic: {z:.4f}')
print(f'Two-sided p-value: {p_value:.4f}')
print('Decision:', 'Reject H0' if p_value < 0.05 else 'Fail to reject H0')

## 4. Confidence interval, relative lift, and effect size

In [ ]:
difference = pB - pA
se_unpooled = np.sqrt(pA*(1-pA)/nA + pB*(1-pB)/nB)
ci_low = difference - 1.96 * se_unpooled
ci_high = difference + 1.96 * se_unpooled
relative_lift = difference / pA
cohen_h = 2*np.arcsin(np.sqrt(pB)) - 2*np.arcsin(np.sqrt(pA))
print(f'Absolute difference: {difference*100:.2f} percentage points')
print(f'Relative lift: {relative_lift:.2%}')
print(f'95% CI: [{ci_low*100:.2f}, {ci_high*100:.2f}] percentage points')
print(f"Cohen's h: {cohen_h:.4f}")

## 5. Chi-square test of independence

In [ ]:
table = np.array([[xA, nA-xA], [xB, nB-xB]])
chi2, chi_p, dof, expected = chi2_contingency(table, correction=False)
print('Contingency table (rows A, B; columns converted, not converted):')
print(table)
print(f'Chi-square statistic: {chi2:.4f}')
print(f'p-value: {chi_p:.4f}')
print('Expected counts:\n', expected)

## 6. Visualize conversion rates

In [ ]:
rates = [pA*100, pB*100]
plt.figure(figsize=(7, 4))
plt.bar(['Control A', 'Variant B'], rates)
plt.ylabel('Conversion rate (%)')
plt.title('Checkout A/B Test: Conversion Rate')
for i, rate in enumerate(rates):
    plt.text(i, rate + 0.2, f'{rate:.2f}%', ha='center')
plt.ylim(0, max(rates)*1.35)
plt.grid(axis='y', alpha=0.25)
plt.show()

## 7. Business interpretation and recommendation

Variant B's observed conversion rate is higher, but statistical tests do not reach the 5% significance threshold. The 95% confidence interval for the difference includes zero. **Recommendation: do not declare B the winner yet. Continue the experiment with a pre-defined sample size/stopping rule, and monitor revenue per visitor and other guardrail metrics.**

## 8. Revenue summary (descriptive only)

In [ ]:
display(summary[['visitors', 'conversions', 'conversion_rate', 'revenue_inr']])
print('Note: revenue differences are descriptive here; this notebook does not test their statistical significance.')